# Inferência do modelo pré-treinado

Continua um código Python a partir de um checkpoint do pré-treino (`train.pretrain`), com a
**gramática do Python** travando a geração (só tokens que mantêm o código sintaticamente válido).

O modelo pré-treinado ainda **não segue instruções**: ele só aprendeu a continuar arquivos de código.
Por isso o texto de entrada é o começo de um arquivo, no formato do pré-treino:

```
<|lang:python|>
<|code|>def fibonacci(n):
<|indent|><|indent|>"""Retorna os n primeiros números de Fibonacci."""
```

e o modelo continua dali até `<|endoftext|>` (ou até o limite de tokens).

## Configuração

In [ ]:
from pathlib import Path

RUN = "teste_20M_tokens"        # nome da rodada passado em --run no pré-treino
CHECKPOINT = None       # None = último step-NNNNN da rodada; ou um caminho específico
DEVICE = "cpu"          # "cpu" ou "xpu" (GPU Intel)

MAX_NEW_TOKENS = 200
TEMPERATURE = 0.7
TOP_P = 0.95
SEED = 0

PREFIXO = '''def fibonacci(n):
    """Retorna os n primeiros números de Fibonacci."""
'''

## Carregar o modelo

In [2]:
import numpy as np
import torch
from transformers import LlamaForCausalLM

from codelm.languages import PROJECT_DATA_DIR
from codelm.tokenizer import load_tokenizer

if CHECKPOINT is None:
    steps = sorted((PROJECT_DATA_DIR / "checkpoints" / RUN).glob("step-*"))
    assert steps, f"nenhum checkpoint step-* em checkpoints/{RUN} (rode o pré-treino antes)"
    CHECKPOINT = steps[-1]

tokenizer = load_tokenizer()
model = LlamaForCausalLM.from_pretrained(CHECKPOINT).to(DEVICE).eval()
print(f"checkpoint: {CHECKPOINT}")
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parâmetros em {DEVICE}")

AssertionError: nenhum checkpoint step-* em checkpoints/small-v1 (rode o pré-treino antes)

## Geração

1. O prefixo vira tokens no formato do pré-treino: cabeçalho (`<|lang:python|>`, `<|code|>`) e o código
   com a indentação relativa. Os `<|dedent|>` que fechariam os blocos no fim são retirados: o código
   ainda não terminou, e o modelo continua no recuo em que o prefixo parou.
2. A gramática do Python consome os tokens do prefixo (se ela recusar um, o prefixo não é Python válido).
3. A cada passo: o modelo dá as notas do próximo token (com cache de K/V, só o token novo é processado),
   a gramática diz quais são permitidos, e `sample` sorteia com temperatura e top-p.

In [ ]:
import time

from codelm.chat_template import CODE, EOS, header_segments
from codelm.grammar import GrammarConstraint
from codelm.indentation import encode_indentation
from codelm.sampler import sample
from codelm.tokenizer import decode_code, encode_segments

EOS_ID = tokenizer.token_to_id(EOS)


def tokens_do_prefixo(codigo: str) -> tuple[list[int], list[int]]:
    """(cabeçalho, código) do prefixo no formato do pré-treino, sem os dedents finais."""
    cabecalho = encode_segments(tokenizer, header_segments("python") + [(True, CODE)])
    corpo = encode_segments(tokenizer, encode_indentation(codigo, "python"))
    while corpo and tokenizer.id_to_token(corpo[-1]) == "<|dedent|>":
        corpo.pop()
    return cabecalho, corpo


@torch.no_grad()
def continuar(codigo: str, com_gramatica: bool = True, seed: int = SEED) -> tuple[str, bool]:
    """Continua o código; devolve (texto completo, se terminou com <|endoftext|>)."""
    rng = np.random.default_rng(seed)
    cabecalho, corpo = tokens_do_prefixo(codigo)
    gramatica = GrammarConstraint("python") if com_gramatica else None
    if gramatica:
        for token in corpo:
            assert gramatica.consume(token), f"a gramática recusou o prefixo em {tokenizer.id_to_token(token)!r}"

    entrada, cache, novos, terminou = torch.tensor([cabecalho + corpo], device=DEVICE), None, [], False
    inicio = time.perf_counter()
    for _ in range(MAX_NEW_TOKENS):
        saida = model(input_ids=entrada, past_key_values=cache, use_cache=True)
        cache = saida.past_key_values
        notas = saida.logits[0, -1].float().cpu().numpy()
        permitidos = gramatica.allowed() if gramatica else None
        token = sample(notas, permitidos, temperature=TEMPERATURE, top_p=TOP_P, rng=rng)
        if gramatica:
            gramatica.consume(token)
        if token == EOS_ID:
            terminou = True
            break
        novos.append(token)
        entrada = torch.tensor([[token]], device=DEVICE)
    duracao = time.perf_counter() - inicio
    print(f"{len(novos)} tokens em {duracao:.1f}s ({len(novos) / duracao:.0f} tok/s), terminou: {terminou}")
    return decode_code(tokenizer, corpo + novos, "python"), terminou

### Com a gramática

In [ ]:
import ast

texto, terminou = continuar(PREFIXO, com_gramatica=True)
print(texto)
if terminou:
    try:
        ast.parse(texto)
        print("\n✔ Python válido (ast.parse)")
    except SyntaxError as erro:
        print(f"\n✘ inválido: {erro}")

### Sem a gramática (para comparar)

Mesmo modelo e mesma semente, só sem a trava.

In [ ]:
texto, terminou = continuar(PREFIXO, com_gramatica=False)
print(texto)
if terminou:
    try:
        ast.parse(texto)
        print("\n✔ Python válido (ast.parse)")
    except SyntaxError as erro:
        print(f"\n✘ inválido: {erro}")

## Dicas

- Troque `PREFIXO` por outro começo de arquivo (imports, uma classe, uma função pela metade...).
- `TEMPERATURE` menor deixa a saída mais previsível; `0` escolhe sempre o token mais provável.
- Para comparar checkpoints ao longo do treino, aponte `CHECKPOINT` para `checkpoints/<rodada>/step-NNNNN`.
- Checkpoints treinados com QAT carregam igual (o QAT não muda os pesos salvos, só como foram treinados).